# South African Retail Cash Cycle 

Rwesana Consultants

August 2025

## Part A: Estimating the Cash Cycle

### 1. Overview of the Monte Carlo Simulation

* The purpose of this Python program is to simulate daily cash flow patterns through tills in large South African supermarkets. It is part of a hybrid statistical model created to understand and estimate cash volumes. 

* Limits on data available and uncertainty around those variables are taken into account. Part B: Literature Review presents the data considered in the model and goes into detail on these aspects.

* The program accomodates three possible scenarios - Conservative, Moderate and Optimistic - each with different input variables. These variables are described in the table below. 

#### Table 1: Model inputs

| Variable                            | Explanation                                                                                                                                                                     |
|-------------------------------------|---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------|
| **Transaction values**            | This average cash transaction value (R212 in moderate case) is core to estimating cash volume flowing through tills. It is based on South African Reserve Bank (SARB) and BankservAfrica data specific to grocery retail. Variability allows modelling different basket sizes. |
| **Cash share of transactions**         | Cash share of total transactions reflects ongoing cash preference in SA retail (about 56% in the moderate case). It drives the scale of cash passing through tills versus card or other payment types.                                           |
| **Till floats per lane**             | Predicted amount of cash kept in each till to provide change and support transactions. Determined partly from international benchmarks and local retail practices, it affects how much cash physically sits in stores and thus influences CIT pickup volumes. |
| **Weekly CIT pickup frequency**   | This frequency (e.g., 3 times per week in moderate case) controls how often cash is collected from the store. Higher frequency reduces till cash accumulation but increases logistical cost; lower frequency increases float and risk.                  |
| **Cashback penetration**            | The percentage of transactions involving cashback (e.g., 11% in the moderate case) reflects how many customers withdraw cash at a till rather than ATMs, a growing trend in SA that influences cash distribution in the cycle.                                  |
| **Cash recycling adoption**         | While not yet applied in this case, this parameter could be explored in future as it will influence efficiency gains and reductions in manual handling if adopted.                                                            |


### 2. Results summary

The simulation is run for the Moderate Case and the main findings are shown below. The volumes are for national daily movements in the retail cash cycle.

#### Table 2: Retail Cash Cycle Volumes, Moderate Case

| Metric                   | Average          | Standard deviation (likely ± variations from the average)    | Predicted range of minimum and maximum possible volumes                         | Interpretation                                                                                   |
|--------------------------|------------------------|--------------------------|-------------------------------------|-------------------------------------------------------------------------------------------------|
| **Daily cash inflow**       | R741.4 million          | R74.5 million            | ~R410 million to R1.02 billion       | Total cash deposited into tills per day across the nation. Variation reflects basket size and transaction variability. |
| **Daily cashback (withdrawals)** | R81.7 million           | R18.4 million            | ~R11.5 million to R156 million       | Cash given out to customers as cashback at tills. Reflects growing use of tills as cash distribution points.                 |
| **Daily change given**       | R185.4 million          | R18.6 million            | ~R103 million to R255 million        | Change returned to customers. Estimated as 25% of total cash inflow, a typical industry proxy.                                 |
| **Net cash in tills**        | R474.4 million          | R50.3 million            | ~R265 million to R674 million        | Remaining cash in tills after cashback and change are dispensed. This represents the on-hand cash before CIT pickups.       |
| **CIT uplift per pickup**    | R158.1 million          | R16.8 million            | ~R88 million to R225 million         | Cash physically collected per pickup, assuming 3 pickups per week (moderate frequency).               |
| **Till float per lane**      | R500 (mean setpoint)    | R75.7                    | ~R188 to R781                       | The cash buffer held in the till for making change and transaction convenience. This variation reflects operational uncertainty.|

### 3. Interpretation

1. **Daily cash inflow (~R741m):**  
   This is the predicted current total cash moving into supermarket tills every day in South Africa, including sales paid in cash. The model predicts typical daily inflows around R740 million, with natural day-to-day variation shown in the standard deviation (likely positive and negative variations from the average) and the predicted range.

2. **Daily cashback (~R82m):**  
   Customers withdrawing cash at retailers are predicted to withdraw roughly R81-82 million daily. This number shows the significant role retailers play in distributing physical cash, supplementing traditional ATMs.

3. **Daily change given (~R185m):**  
   An estimated 25% of the cash inflow is returned as change. This cash quickly re-enters the retail cycle through subsequent transactions and likely makes up a large proportion of the cash movement between consumers and store till.

4. **Net cash in tills (~R474m):**  
   Cash that remains physically inside tills at the end of the day after dispensing cashback and change. This on-hand cash is the amount vulnerable to theft, shrinkage, and awaiting CIT collection.

5. **CIT uplift per pickup (~R158m):**  
   Given a moderate CIT schedule (3 pickups per week), each pickup collects roughly R158 million in cash from retailer vaults or central stores. This amount will likely be split among multiple store locations and vehicles but illustrates the scale of cash logistics required.

6. **Till float per lane (~R500 average):**  
   Represents the cash buffer per till lane. This float ensures smooth operation, enabling accurate change and customer service. The range accounts for uncertainty and operation variability.

In [1]:
import numpy as np
import pandas as pd
np.random.seed(42)

In [2]:
# Define scenario parameter ranges based on research and benchmarking
params = {
    "avg_cash_transaction": {"conservative": 180, "moderate": 212, "optimistic": 250},  # ZAR
    "cash_share_transactions": {"conservative": 0.50, "moderate": 0.56, "optimistic": 0.65},  # fraction
    "till_float_per_lane": {"conservative": 300, "moderate": 500, "optimistic": 800},  # ZAR
    "cit_pickup_frequency_per_week": {"conservative": 7, "moderate": 3, "optimistic": 2},  # pickups per week
    "cashback_penetration": {"conservative": 0.05, "moderate": 0.11, "optimistic": 0.20},  # fraction
    "cash_recycling_adoption": {"conservative": 0.00, "moderate": 0.15, "optimistic": 0.40}  # fraction, for future use
}

In [3]:
def monte_carlo_simulation(n_sim=10000, scenario='moderate'):
    """
    Monte Carlo simulation for cash flows in SA supermarket tills. Each simulation run 
    samples possible transaction sizes and cashback rates. It computes daily cash inflow, 
    cashback, change given out, net till cash after withdrawals and change, and required 
    CIT uplift per pickup. 
    
    Simulation output gives mean, standard deviation (SD), min, max, and percentiles for 
    each metric. Adjust the scenario in the function call for the moderate, conservative or 
    optimistic case. Adjust main modelling assumptions as needed.

    Model outputs:
    - sim_results: full simulated samples (10,000 by default)
    - sim_summary: summary statistics 

    """

    # Scenario parameters
    avg_tx = params['avg_cash_transaction'][scenario]
    cash_share = params['cash_share_transactions'][scenario]
    till_float = params['till_float_per_lane'][scenario]
    cit_freq = params['cit_pickup_frequency_per_week'][scenario]
    cashback_frac = params['cashback_penetration'][scenario]

    # Main modelling assumptions
    # Number of cash transactions per day at national level
    daily_cash_transactions = 3_500_000  # Approx 104 million/month ÷ 30 days

    # Simulate transaction values across a normal distribution with 10% SD and non-negative values
    tx_values = np.random.normal(loc=avg_tx, scale=avg_tx*0.10, size=n_sim)
    tx_values = np.clip(tx_values, a_min=0, a_max=None)

    # Simulate proportion of transactions with cashback across a normal distribution, 20% SD between 0 and 1
    cashback_ratios = np.random.normal(loc=cashback_frac, scale=cashback_frac*0.20, size=n_sim)
    cashback_ratios = np.clip(cashback_ratios, a_min=0, a_max=1)

    # Average change given out as 25% of amount transacted
    change_ratio = 0.25

    # Simulate 
    daily_cash_inflow = daily_cash_transactions * tx_values
    daily_cashback = daily_cash_inflow * cashback_ratios
    daily_change_given = daily_cash_inflow * change_ratio
    net_cash_in_till = daily_cash_inflow - (daily_cashback + daily_change_given)
    cit_uplift_per_pickup = net_cash_in_till / cit_freq
    till_float_sim = np.random.normal(loc=till_float, scale=till_float*0.15, size=n_sim)
    till_float_sim = np.clip(till_float_sim, a_min=0, a_max=None)

    # Compile output
    results = pd.DataFrame({
        'Daily cash inflow': daily_cash_inflow,
        'Daily cashback (retail withdrawal)': daily_cashback,
        'Daily change given': daily_change_given,
        'Net cash in tills': net_cash_in_till,
        'CIT uplift per pickup': cit_uplift_per_pickup,
        'Till float': till_float_sim
    })

    summary = results.describe(percentiles=[0.05, 0.25, 0.5, 0.75, 0.95])
    return results, summary

In [4]:
sim_results, sim_summary = monte_carlo_simulation(n_sim=10000, scenario='moderate')
display(sim_summary)
sim_summary.to_csv('C:/Users/36050/.venv/MC_simulation_moderate.csv')

,Daily cash inflow,Daily cashback (retail withdrawal),Daily change given,Net cash in tills,CIT uplift per pickup,Till float
count,1.000000e+04,1.000000e+04,1.000000e+04,1.000000e+04,1.000000e+04,10000.000000
mean,7.418415e+08,8.180943e+07,1.854604e+08,4.745717e+08,1.581906e+08,499.065292
std,7.445691e+07,1.830606e+07,1.861423e+07,5.050874e+07,1.683625e+07,74.356362
min,4.509579e+08,1.983642e+07,1.127395e+08,2.827931e+08,9.426437e+07,225.868511
5%,6.192095e+08,5.319570e+07,1.548024e+08,3.927667e+08,1.309222e+08,378.727040
25%,6.920938e+08,6.914057e+07,1.730234e+08,4.401836e+08,1.467279e+08,447.488606
50%,7.418075e+08,8.109845e+07,1.854519e+08,4.735082e+08,1.578361e+08,499.567382
75%,7.917942e+08,9.382474e+07,1.979486e+08,5.083014e+08,1.694338e+08,549.792307
95%,8.638889e+08,1.129097e+08,2.159722e+08,5.592547e+08,1.864182e+08,622.577737
max,1.033327e+09,1.501990e+08,2.583317e+08,6.734121e+08,2.244707e+08,776.871838


## Part B: Literature Review

###  1. Local-Level Data Collection

This part of the report presents publicly available data identified for modelling the South African retail cash cycle, with specific focus on large supermarket chains, cash volumes at tills, and cashback operations. The data sources provide both direct measurements and proxy indicators that will enable construction of conservative, moderate, and upper-end estimates for cash flows through the retail system.

The data is consolidated into a structured analytical framework, covering market structure, payment patterns, cash circulation, demographics, and operational metrics for the South African supermarket sector.

| Supermarket Chain                | Description                                          | Reported Turnover (2023) |
|----------------------------------|------------------------------------------------------|---------------------------|
| **Shoprite Holdings**            | 3,300+ convenience stores, discounters, hypermarkets, traditional supermarkets. | R215 billion              |
| **Pick n Pay**                   | 2,000+ supermarkets, hypermarkets, discount outlets, liquor and clothing stores. | R106 billion              |
| **SPAR Group**                   | 1,000+ food retail, liquor and pharmaceutical stores. | R91 billion               |
| **Woolworths Holdings Limited**  | 470+ stores in premium market segment.              | R73.2 billion             |
| **Massmart Holdings**            | 411+ general merchandise, liquor, home improvement, and wholesale food markets. | R84.9 billion (2021)      |

#### 1.1 Cash Usage Patterns in South African Retail

According to Cash Connect operations data, as many as 90% of transactions in SAare still settled with cash, although it is difficult to be certain of this due to the large informal sector. Despite cash being used in so many transactions, it represents only about 20% of total payment value, highlighting its prevalence in smaller-value transactions typical of retail environments. This data suggests substantial cash volumes flowing through retail tills, even though individual transactions may be relatively modest.

The total amount of cash circulating in SAhas reached R182 billion, according to BankservAfrica. The South African Reserve Bank's (SARB) Payments Study Report shows that cash payments averaged R208 per transaction in 2023, lower than the overall average payment value of R529 across all payment methods. 

Provincial analysis reveals significant variations. Gauteng accounts for 30% of payment volume but only 24% of payment value, with an average payment value of R725. KwaZulu-Natal represents 18% of volume and 19% of value (R493.50 average), while the Western Cape accounts for 13% of volume and 15% of value (R511 average). These provincial differences will be crucial for developing regional cash flow estimates within the model.

#### 1.2 Supermarket Cash Volumes and Transaction Data

BankservAfrica's point-of-sale (POS) transaction data reveals the scale of supermarket cash operations. In December 2023, total spending at grocery stores and supermarkets reached R50 billion, nearly doubling from December 2022. This dramatic increase reflects both inflation and increased transaction volumes, with grocery stores and supermarkets representing the largest single category of consumer spending. Such volume data, combined with spending values, provides insight into average transaction sizes and cash handling requirements.

Shoprite reported merchandise sales growth of 16% to R215 billion for June 2023 to July 2024. The company's core Supermarkets RSA segment achieved sales growth of 17.8%, with like-for-like sales up 10.3%. Checkers and Checkers Hyper divisions grew 18%, while Shoprite and Usave grew 15.6%. These growth rates suggest increasing cash flows through retail systems.

Pick n Pay demonstrated recovery with like-for-like sales growth improving from -0.5% in FY 2024 to +3.6% in FY 2025. The company's turnover rose 5.6% for the 53-week period ended March 2025. SPAR Group achieved combined turnover growth of 3.6%. These performance metrics indicate the dynamic nature of cash flows that the model must capture.

#### 1.3 Cashback Operations and Volumes

Till-based cashback represents a critical component of the retail cash cycle, though specific volume data remains limited due to competitive sensitivity. However, several proxy indicators and market research provide insight into cashback operations and their growth trajectory.

Standard Bank data provides direct evidence of till-based cashback growth, noting a 100% surge in cashback transactions at retail chain POS since 2019. This dramatic increase suggests retailers are increasingly serving as cash distribution points, supplementing traditional ATM networks. Standard Bank, as South Africa's largest bank by assets, reports significant customer migration toward cashback at POS as an alternative to ATM withdrawals. 

Major retailers have integrated cashback services into their operations as part of broader financial services offerings. Shoprite operates Money Market counters providing money transfer services estimated to hold 70% to 80% of the South African money transfer market. These counters also facilitate cashback transactions as part of their financial services. FNB's partnership with Pick n Pay through the e-Bucks program demonstrates how financial institutions are leveraging retail networks for customer engagement and potentially cash distribution.

#### 1.4 Local Data Sources Assessment

The compiled data originates from multiple sources providing different perspectives on South African retail cash operations. BankservAfrica serves as the banking sector's official clearing partner and payment system operator, providing high-confidence data on cash circulation and transaction volumes. The SARB Payments Study Report offers the most statistically robust insights into cash usage patterns.

Corporate annual reports from major retailers provide audited financial data on turnover and transaction volumes. These sources offer reliable baseline data for estimating cash flows, though they typically do not disaggregate cash versus card transactions. Industry research reports from ResearchAndMarkets.com provide market sizing data for cashback programs, though these focus on loyalty programs rather than till-based cash withdrawals.

Trade publications and news sources provide operational insights and trend analysis, though these require careful validation against primary, more authoritative data sources. The combination of regulatory data, corporate reporting, and market research provides a triangulated view of retail cash operations suitable for modelling purposes.

This initial data collection establishes the foundation for developing a hybrid modelling approach, with sufficient quantitative data to support the conservative, moderate and upper-end estimates of cash flows through South African supermarket operations. The identified data sources will enable construction of a comprehensive model capturing cash inflows to tills, change dispensed, cashback provided, and cash storage requirements at retail processing centres.

### 2. National Payments Patterns and Cash-Volume Proxies (2023-2025)

This section considers macro-level evidence that helps triangulate how much cash actually moves through large SA food retailers’ tills and their in-store cash-processing centres.

#### 2.1 How South Africans Pay for Groceries

| Metric  | Cash | Debit card | Credit card | Notes |
|---|---|---|---|---|
| Share of all retail payments – volume | 56% | 34% | 1.9% | Cash remains the single biggest payment method by count. |
| Share of all retail payments – value | 21% | 55% | 4.3% | Even though cash dominates volumes, higher-value baskets increasingly move to cards. |
| Average transaction value (all methods) | R 52 | — | — | Bench-mark for later modelling. |
| Average cash transaction value | R 20 | — | — | Confirms cash is used mostly for low-ticket items. |
| Average cash transaction – groceries | R 21 | — | — | Proxy for the *till-level* change-making float. |
| Average debit-card transaction – groceries | — | R 71 | — | About 3.4x the cash basket – useful in split-payment modelling. |

**Implication for tills**  

With 31% of recorded payments by volume going to groceries, the cash share converts to roughly:

0.56 (cash volume) × 0.31 (grocery share) ≈ 17% of all consumer payments being *cash at supermarket tills*. 

That is the *single largest identifiable cash-handling node in the economy*.

#### 2.2 Physical Cash Demand Spikes

| Indicator | 2022 (R bn) | 2023 (R bn)| 2024  (R bn) | Comment |
|---|---|---|---|---|
| BankservAfrica ICMS cash orders, December | 84 | 81 | 87 | Up 4% y/y despite digital-payment growth – retailers order most of this stock for tills and back-office cash centres. |
| One-day peaks within December | 8 (15 Dec 2022) | 6.7 (13 Dec 2024) | — | Correlates with public-holiday grocery rush. |

#### 2.3 Cash Access and Recycling Channels

| Channel | Relevance to supermarkets |
|---|---|
| ATM withdrawals | Competing source of notes consumers later spend in store. 1.26 bn transactions in 2023; forecast 1.34 bn by 2028. |
| POS cash-back at retailers | Converts electronic value into till float on-site; Standard Bank reports >100% surge in cashback volumes since 2019 and **11% of all its cash withdrawals now happen at supermarket check-outs**. |
| Retail till cash-deposit partnerships | Shoprite/Checkers accept up to R3 000 deposits at 1 700 stores, flat R19.95 fee; flows straight into store vaults before Cash-in-Transit (CIT) uplift. |

#### 2.4 Costs and Risks that Keep Cash in the System

-   Cash remains the cheapest payment option for many low-income consumers — **90% of South African transactions under R100 are still cash**.  
-   SARB estimates the national cash-handling ecosystem costs ~R88 bn per year; retailers carry ~40% of those costs through CIT fees, shrinkage, and insurance.

#### 2.5 Aggregate Daily and Weekly Cash Throughput (Derivable Inputs)

- Adult population ≥ 18 years: 40.5 million (m) people.  
- Median number of payments per adult:  
    15 per month → 610 m payments/month. 
- Grocery share:   
    31% → 189 m grocery payments/month. 
- Cash share of grocery payments: 
    55% of volume × 31% grocery share = 17% of total → 104 m cash grocery payments/month.  

- **Average cash grocery basket: R212.**
- **Monthly till cash takings for large supermarkets = ~R22 bn (104 m × R212).**
- **Weekly average ≈ R5.1 bn.**
- **Daily average ≈ R730 m.** 
  
These figures provide a national baseline for the deterministic segment of the cash-cycle model. The consolidated macro-statistics, together with the retailer turnover figures, are the raw-data foundation for the South African supermarket cash-cycle model. They quantify:

-   The *inflow* of notes and coins at checkout.  
-   The *outflow* via change and cashback.  
-   The periodic *surges* that strain tills and cash-processing centres.

#### 2.6 Missing Data

1. **Retailer-specific cash splits.** Annual reports give total sales but seldom disclose tender mix. Industry interviews or POS data panels will be explored.  
2. **CIT uplift cadence and vault dwell time.** SBV Services white paper cites R1 trn cash moved annually across 25 cash centres. Distribution between food retail and other sectors will be useful to know.  
3. **Provincial variance.** Gauteng alone handles 24% of payment value yet only 30% of volume. A scaling factor will refine sub-national modelling.

To handle these unknowns, incomplete data, and the randomness inherent in human behaviour, the model will incorporate stochastic uncertainty parameters.

### 3. International Benchmarks and Best Practices for Retail Cash Cycle Models

This section is the final component of the data foundation: international benchmarks, industry standards, and cash management best practices that can inform model parameters and validate the approach.

#### 3.1 International Cash Conversion Cycle Benchmarks

**Grocery/Food Retail Sector Performance Standards**

| Metric | Best-in-Class | Industry Average | Below Average | Sources |
|---|---|---|---|---|
| Cash Conversion Cycle (CCC) | 10-20 days | 45 days | 60+ days | Multiple industry studies |
| Days Inventory Outstanding (DIO) | 15-25 days | 50-70 days | 85+ days | PWC Working Capital Study 2017 |
| Days Sales Outstanding (DSO) | 1-3 days | 15-17 days | 25+ days | Reflects immediate payment at checkout |
| Days Payable Outstanding (DPO) | 25-45 days | 15-20 days | 10-15 days | Supplier payment terms |

**Insight for the SA Model**

- **Fast-Moving Consumer Goods (FMCG) companies can achieve negative CCC** (-24 days average), meaning they collect from customers before paying suppliers - a powerful cash generation model achievable by major SA supermarket chains.

#### 3.2 Cash Float Management Standards

**International Till Float Benchmarks**

From global retail cash management studies and industry practices:

| Store Format | Typical Float per Till | Notes |
|---|---|---|
| Convenience stores | $150-200 | Lower transaction values |
| Supermarkets | $200-500 | Depends on average transaction size |
| Hypermarkets | $500-800 | Higher-value transactions |
| Premium retailers | $300-1000 | Boutique format |


**Float Optimisation Principles**

- Float should approximate **1-2 times average transaction value**
- **Frequent pick-ups** when till cash exceeds R1,500 (SA practice).
- **Denomination management**: Proper mix of notes and coins for efficient change-making.

#### 3.3 Cash Recycling and Automation Benefits

**Operational Efficiency Gains from International Implementations**

| Technology | Time Savings | Cost Reduction | ROI Period |
|---|---|---|---|
| Cash recyclers (grocery) | 8+ hours daily per store | 30-40% cash handling costs | 12-18 months |
| Smart safes | 2-4 hours daily | 15-25% reduction | 18-24 months |
| Automated cash management | Up to 40% in time/money | Significant shrinkage reduction | Variable |

#### 3.4 CIT and Security Benchmarks

**International CIT Cost Structures**

| Component | SA Average | International Range | Best Practice |
|---|---|---|---|
| CIT service fees | R3,900/month (6x weekly) | $2,000-5,000/month | Optimise frequency based on volume |
| Cash insurance (in-store) | 1.25% of exposure | 0.8-2.0% globally | Risk-based pricing |
| CIT insurance | 0.08% of exposure | 0.05-0.15% globally | Volume discounts available |
| Cash shrinkage | 0.15% of turnover | 0.1-0.3% globally | Automation reduces significantly |


#### 3.5 Technology and Process Innovation

**Global Cash Management Evolution**

| Innovation Area | Current Practice | Emerging Trends | SA Relevance |
|---|---|---|---|
| ATM integration | Separate networks | In-store ATM recyclers | High - major retailers exploring |
| Cash deposit services | Bank branches | Till-point deposits | Implemented - Shoprite/Checkers |
| Real-time monitoring | Daily reconciliation | Continuous cash tracking | Growing adoption |
| Predictive analytics | Historical patterns | AI-driven cash forecasting | Early stage |

#### 3.6 Regulatory and Compliance Framework

**International Cash-Handling Standards**

- **EU Late Payment Directive**: Impacted retailer DPO by -9.3 days industry-wide
- **SABS Cat 4 standards**: South African security requirements for cash vaults
- **Treasury Single Account**: Government cash management best practices applicable to corporate cash centralisation

### 4. Model Calibration Implications

**Derived Parameters for SA Supermarket Model**

Based on benchmarking, the following ranges provide realistic bounds for the Monte Carlo simulation:

| Parameter | Conservative | Moderate | Optimistic | Basis |
|---|---|---|---|---|
| Average transaction (cash) | R180 | R212 | R250 | SARB data + inflation |
| Cash share of transactions | 50% | 56% | 65% | Current SA levels |
| Till float per lane | R300 | R500 | R800 | International + SA practices |
| CIT pickup frequency | Daily | 3x weekly | 2x weekly | Volume-dependent |
| Cashback penetration | 5% | 11% | 20% | Standard Bank data |
| Cash recycling adoption | 0% | 15% | 40% | Technology diffusion curve |

#### 4.1 Validation Framework

**Benchmarking the Model Outputs**

The international data provides the following validation checkpoints:

1. **Total cash cycle time**: Should align with 10-45 day range for food retail.
2. **Working capital efficiency**: Negative CCC achievable for major chains.
3. **Operational costs**: Automation can reduce cash handling by 30-40%.
4. **Technology ROI**: 12-24 month payback periods realistic.

### 4.2 Conclusion

**Hybrid Model Components**

The international best practices suggest the SA model should incorporate:

- **Deterministic foundation**: Based on SARB payment data and retailer financials.
- **Stochastic overlays**: Monte Carlo simulation for demand variability, technology adoption, and market evolution.
- **Scenario planning**: Conservative, moderate, and optimistic cases reflecting different adoption rates of cash automation.
- **Technology roadmap**: Progressive integration of cash recycling and smart safe technologies.
- **Regulatory sensitivity**: Ability to model impact of payment regulation changes.

In conclusion, we now have sufficient quantitative inputs to build a robust model of the South African supermarket cash cycle that can serve both analytical and strategic planning purposes.

### Main Data Sources

- South African Reserve Bank Payments Study Report 2023 [1][2]  
Provides authoritative data on cash usage patterns, transaction values, and payment method splits specific to South Africa

- BankservAfrica Integrated Cash Management Service (ICMS) Data [3][4][5]  
Quantifies actual cash circulation volumes (R84-87.7 billion in December periods) and seasonal demand patterns

- Standard Bank Cash Transaction Analysis [6][7]  
Documents 100% surge in cashback transactions since 2019 and reveals 11% of cash withdrawals now occur at retail POS

- Shoprite Holdings Annual Financial Reports [8][9]  
Provides scale data for SA's largest retailer (R215 billion turnover, 3,300 outlets, 80% of sales from core supermarket brands)

- SBV Consumer Cash Survey White Paper [10]  
Confirms 97% of SMME retailers and consumers remain equally reliant on cash as payment mechanism

- PWC Working Capital in Retail Sector Study 2017 [11]  
International benchmark showing food retail working capital ranges from -11.7 to +85.8 days, with 56.8 day average

- European Central Bank Cash Management Simulation Model [12][13]  
Provides validated international framework for modeling cash payment choices and management decisions

- SARB Digital Payments Roadmap [14]  
Shows growth in retail payment transaction volumes and digital payment adoption trends affecting cash demand

- Competition Commission Grocery Retail Market Inquiry Final Report [15]  
Comprehensive analysis of SA grocery retail sector structure, market shares, and competitive dynamics

- Cash Connect/Connected Group Cash Management Solutions Data [16][17][18]  
Quantifies cash handling cost savings (up to 40%) and operational benefits of automated cash management in SA retail context

These sources collectively provided:

- **Authoritative payment behavior data** (sources 1, 2, 3)
- **Retailer-specific scale and financial metrics** (sources 4, 9)
- **Cash dependency validation** (sources 5, 8)
- **International benchmarking frameworks** (sources 6, 7)
- **Operational cost and efficiency data** (source 10)

This combination enables construction of both the deterministic baseline model and the stochastic uncertainty parameters needed for the Monte Carlo simulation approach.

[1] https://www.resbank.co.za/content/dam/sarb/publications/media-releases/2024/payments/SARB%20Payments%20Study%20Report%202023%20Executive%20Summary.pdf

[2] https://www.resbank.co.za/content/dam/sarb/what-we-do/payments-and-settlements/payments-insights/SARB%20Payments%20Study%20Report%202023%20Executive%20Summary.pdf

[3] https://it-online.co.za/2023/01/31/consumers-splurged-on-groceries-entertainment-and-fuel-in-december/

[4] https://www.moonstone.co.za/holiday-season-spending-soars-cash-remains-king-amid-festive-shopping-frenzy/

[5] https://www.engineeringnews.co.za/article/consumers-spent-most-money-on-groceries-entertainment-fuel-and-food-in-dec-2023-01-30

[6] https://www.standardbank.co.za/southafrica/news-and-media/newsroom/as-the-shift-to-digital-accelerates-who-is-still-relying-on-cash

[7] https://www.standardbank.co.za/southafrica/news-and-media/newsroom/standard-bank-saves-its-customers-more-than-r670-million-in-3-years

[8] https://www.shopriteholdings.co.za/docs/shp-ir-2024.pdf

[9] https://www.shopriteholdings.co.za/docs/shp-afs-2024.pdf

[10] https://www.sbv.co.za/wp-content/uploads/2024/03/SBV-Consumer-Cash-Survey-White-Paper.pdf

[11] https://www.pwc.ch/en/publications/2017/working-capital-retail-study-2017.pdf

[12] https://www.ecb.europa.eu/pub/pdf/scpwps/ecbwp1874.en.pdf

[13] https://www.econstor.eu/bitstream/10419/154307/1/ecbwp1874.pdf

[14] https://www.resbank.co.za/content/dam/sarb/what-we-do/payments-and-settlements/regulation-oversight-and-supervision/Digital%20Payments%20Roadmap.pdf

[15] https://www.compcom.co.za/wp-content/uploads/2019/12/GRMI-Non-Confidential-Report.pdf

[16] https://bizmag.co.za/automated-cash-management-south-african-retailers/

[17] https://it-online.co.za/2024/10/21/high-cash-usage-drives-a-need-for-automated-cash-management/

[18] https://www.connected.co.za/media-releases/latest-news-cashconnect/1254-sharpening-your-retail-business-s-competitive-edge-with-tighter-cash-flow-management.html